# InfraRisk AI - Component 3: Exploratory Data Analysis (Infrastructure)

This notebook performs Exploratory Data Analysis (EDA) on the infrastructure project data generated by the `WorldBankLoader` in `src.data.world_bank_loader`.
We analyze:
- Project distribution across sectors, subsectors, and regions.
- Financial splits (Debt vs. Equity) and leverage ratios.
- Concession lengths, credit ratings, and project statuses.
- Debt Service Coverage Ratios (DSCR) and correlations with macroeconomic indicators.

We generate 15+ publication-quality interactive Plotly visualizations.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Adjust sys.path to import src
sys.path.append(os.path.abspath('..'))
from src.data.world_bank_loader import WorldBankLoader

# Setup directory
os.makedirs('../data', exist_ok=True)

## 1. Load and Verify Infrastructure & WDI Data

In [ ]:
# Ingest data using WorldBankLoader
loader = WorldBankLoader(cache_dir='../data')
if not os.path.exists('../data/world_bank_combined.csv'):
    print("Generating new combined World Bank PPI & WDI dataset...")
    df = loader.get_combined_dataset(10000)
else:
    print("Loading cached dataset...")
    df = pd.read_csv('../data/world_bank_combined.csv')

print(f"Dataset loaded. Shape: {df.shape}")

In [ ]:
# Basic statistical summary
df.info()

In [ ]:
df.describe()

## 2. Interactive Visualizations (15+ Publication-Quality Plotly Charts)

### Viz 1: Distribution of Project Investment Values (USD Millions)

In [ ]:
fig = px.histogram(df, x="investment_value_usd_m", nbins=50, marginal="box",
                   title="Distribution of Project Investment Values (USD M)",
                   labels={"investment_value_usd_m": "Investment Value (USD M)"},
                   template="plotly_white", color_discrete_sequence=["#1f77b4"])
fig.update_layout(bargap=0.1)
fig.show()

### Viz 2: Debt vs. Equity Split by Sector

In [ ]:
fig = px.scatter(df.sample(2000, random_state=42), x="equity_value_usd_m", y="debt_value_usd_m",
                 color="sector", trendline="ols",
                 title="Debt vs. Equity Value by Sector (Sample of 2,000 Projects)",
                 labels={"equity_value_usd_m": "Equity Value (USD M)", "debt_value_usd_m": "Debt Value (USD M)"},
                 template="plotly_white")
fig.show()

### Viz 3: Debt-to-Equity Ratio Distribution by Sector

In [ ]:
fig = px.box(df, x="sector", y="debt_equity_ratio", color="sector",
             title="Debt-to-Equity Ratio Distribution by Sector",
             labels={"debt_equity_ratio": "Debt-to-Equity Ratio", "sector": "Sector"},
             template="plotly_white")
fig.show()

### Viz 4: Concession Length by Sector

In [ ]:
fig = px.violin(df, x="sector", y="concession_period_years", color="sector", box=True, points="outliers",
                title="Concession Period (Years) Distribution by Sector",
                labels={"concession_period_years": "Concession Period (Years)", "sector": "Sector"},
                template="plotly_white")
fig.show()

### Viz 5: Project Status Count by Region

In [ ]:
status_region = df.groupby(["region", "status"]).size().reset_index(name="count")
fig = px.bar(status_region, x="region", y="count", color="status", barmode="group",
             title="Project Count by Region and Status",
             labels={"count": "Number of Projects", "region": "Region", "status": "Status"},
             template="plotly_white")
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### Viz 6: Total Investment Flows over Time by Sector

In [ ]:
inv_time = df.groupby(["financial_closure_year", "sector"])["investment_value_usd_m"].sum().reset_index()
fig = px.line(inv_time, x="financial_closure_year", y="investment_value_usd_m", color="sector", markers=True,
              title="Total Project Investment Value by Year and Sector",
              labels={"investment_value_usd_m": "Total Investment (USD M)", "financial_closure_year": "Financial Closure Year"},
              template="plotly_white")
fig.show()

### Viz 7: Sovereign Rating vs. Project Status

In [ ]:
rating_status = df.groupby(["sovereign_rating", "status"]).size().reset_index(name="count")
rating_order = ["AAA", "AA", "A", "BBB+", "BBB", "BBB-", "BB+", "BB", "BB-", "B+", "B", "B-", "CCC+"]
rating_order = [r for r in rating_order if r in df["sovereign_rating"].unique()]

fig = px.bar(rating_status, x="sovereign_rating", y="count", color="status",
             category_orders={"sovereign_rating": rating_order},
             title="Project Status Breakdown by Sovereign Credit Rating",
             labels={"count": "Number of Projects", "sovereign_rating": "Sovereign Rating"},
             template="plotly_white")
fig.show()

### Viz 8: Average DSCR by Sovereign Rating

In [ ]:
avg_dscr = df.groupby("sovereign_rating")["dscr"].mean().reset_index()
fig = px.bar(avg_dscr, x="sovereign_rating", y="dscr",
             category_orders={"sovereign_rating": rating_order},
             title="Average Debt Service Coverage Ratio (DSCR) by Sovereign Rating",
             labels={"dscr": "Average DSCR", "sovereign_rating": "Sovereign Rating"},
             template="plotly_white", color_discrete_sequence=["#2ca02c"])
fig.add_hline(y=1.0, line_dash="dash", line_color="red", annotation_text="DSCR = 1.0 (Breakeven)")
fig.show()

### Viz 9: DSCR Distribution by Project Status

In [ ]:
fig = px.box(df, x="status", y="dscr", color="status",
             title="DSCR Distribution by Project Status",
             labels={"dscr": "Debt Service Coverage Ratio (DSCR)", "status": "Status"},
             template="plotly_white")
fig.add_hline(y=1.0, line_dash="dash", line_color="red")
fig.show()

### Viz 10: Government Grant Value Distribution

In [ ]:
grant_df = df[df["government_grant_usd_m"] > 0]
fig = px.histogram(grant_df, x="government_grant_usd_m", nbins=30, marginal="rug",
                   title="Distribution of Government Grants (For Supported Projects)",
                   labels={"government_grant_usd_m": "Government Grant (USD M)"},
                   template="plotly_white", color_discrete_sequence=["#e377c2"])
fig.show()

### Viz 11: Correlation Heatmap of Project & Macroeconomic Variables

In [ ]:
num_cols = ["investment_value_usd_m", "debt_value_usd_m", "equity_value_usd_m", 
            "debt_equity_ratio", "concession_period_years", "dscr", 
            "gdp_growth", "inflation", "real_interest_rate", 
            "regulatory_quality", "rule_of_law", "government_effectiveness"]
corr = df[num_cols].corr()
fig = px.imshow(corr, text_auto=".2f", aspect="auto",
                title="Correlation Matrix of Infrastructure Project and Macroeconomic Variables",
                color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
fig.show()

### Viz 12: Top 10 Sponsors by Cumulative Project Investment

In [ ]:
top_sponsors = df.groupby("sponsors")["investment_value_usd_m"].sum().reset_index()
top_sponsors = top_sponsors.sort_values(by="investment_value_usd_m", ascending=False).head(10)
fig = px.bar(top_sponsors, x="investment_value_usd_m", y="sponsors", orientation="h",
             title="Top 10 Sponsors by Cumulative Project Investment Value (USD M)",
             labels={"investment_value_usd_m": "Cumulative Investment Value (USD M)", "sponsors": "Sponsor Group"},
             template="plotly_white", color_discrete_sequence=["#bcbd22"])
fig.update_layout(yaxis={'categoryorder':'value ascending'})
fig.show()

### Viz 13: Subsector Investment Treemap

In [ ]:
fig = px.treemap(df, path=["sector", "subsector"], values="investment_value_usd_m",
                 title="Treemap of Project Sectors and Subsectors by Total Investment",
                 template="plotly_white")
fig.show()

### Viz 14: Sovereign Rating vs. Debt-to-Equity Ratio

In [ ]:
fig = px.violin(df, x="sovereign_rating", y="debt_equity_ratio", color="sovereign_rating",
                category_orders={"sovereign_rating": rating_order},
                title="Debt-to-Equity Ratio Distribution by Sovereign Credit Rating",
                labels={"debt_equity_ratio": "Debt-to-Equity Ratio", "sovereign_rating": "Sovereign Rating"},
                template="plotly_white")
fig.show()

### Viz 15: Average Concession Period vs. Sovereign Rating

In [ ]:
avg_concession = df.groupby("sovereign_rating")["concession_period_years"].mean().reset_index()
fig = px.line(avg_concession, x="sovereign_rating", y="concession_period_years", markers=True,
              category_orders={"sovereign_rating": rating_order},
              title="Average Concession Period (Years) by Sovereign Rating",
              labels={"concession_period_years": "Average Concession Period", "sovereign_rating": "Sovereign Rating"},
              template="plotly_white")
fig.show()

### Viz 16: Annual Project Count by Sector

In [ ]:
proj_counts = df.groupby(["financial_closure_year", "sector"]).size().reset_index(name="count")
fig = px.area(proj_counts, x="financial_closure_year", y="count", color="sector",
              title="Annual Project Additions by Sector",
              labels={"count": "Number of Projects", "financial_closure_year": "Financial Closure Year"},
              template="plotly_white")
fig.show()

### Viz 17: Geographical Project Locations

In [ ]:
df["log_investment"] = np.log1p(df["investment_value_usd_m"])
fig = px.scatter_geo(df.sample(1000, random_state=42), lat="latitude", lon="longitude", color="sector",
                     size="log_investment", hover_name="project_name",
                     title="Geographical Distribution of Projects (Sample of 1,000 Projects)",
                     labels={"log_investment": "Log(Investment)"},
                     template="plotly_white")
fig.show()